# Binary field square

Adapted from Qualtran v0.7.0 at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0. Complete concrete and symbolic examples use FlagQuantum metadata, resource graphs and signature tables. Native table layout, operation names and resource labels replace the original presentation.


In [ ]:
from IPython.display import SVG, display
from examples.algorithms.field_square_catalog import field_square_examples, field_square_example_graph
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

## Frobenius powers and their inverse

For a binary field of degree m, the mathematical operation is |x> -> |x^(2**count)>. The original prose prints a^(2k), while its value callback uses 2**k. Repeated squaring is a linear binary transformation, implemented with CNOTs and a logical output permutation.

`binary_field_encoding` resolves the field descriptor or polynomial. `gf2_frobenius(target, count, modulus=..., adjoint=...)` accepts one big-endian target word. The count is a nonnegative concrete integer. `adjoint=True` applies the mathematical inverse. The original k/uncompute parameters correspond to count/adjoint. The register x is THRU and no additional quantum or classical workspace is allocated. Symbolic m is retained in the signature and the m(m-1) PLU upper bound.

`field_frobenius_matrix` exposes the mathematical matrix with least-significant coefficient first. The executable chain reverses the big-endian word into this order, applies the shared PLU component, and restores the logical word order. `bind_field_outputs` explicitly fixes an output permutation with separately counted SWAPs when the consumer needs fixed physical wires.

The unchanged reference has known k=0 and non-power-of-two matrix discrepancies. Its linear component's adjoint decomposition also emits the forward stream. Existing source witnesses remain in the audit. Native mathematical powers and exact inverse behavior are distinguished from the forward k=1/2/4 source-parity evidence.


In [ ]:
from flagquantum.simulation.finite_field_addition import binary_field_encoding
from flagquantum.simulation.finite_field_frobenius import field_frobenius_matrix, field_frobenius_classical_value
from flagquantum.compiler.registers.finite_field.field_linear import gf2_frobenius
from flagquantum.compiler.registers.finite_field.field_multiplication import bind_field_outputs
from examples.algorithms.binary_field_arithmetic import FieldOperation
import flagquantum as fq
import torch

### Example Instances

In [ ]:
examples = field_square_examples()
concrete = examples[0]
concrete

In [ ]:
symbolic = examples[1]
symbolic

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg(tuple((name, signature, {}) for name, signature, _ in examples))))

### Call Graph

In [ ]:
graph = field_square_example_graph(concrete)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
print("One-level declared leaves", graph.leaf_counts)
print("Concrete PLU source model", concrete[2])
print("Symbolic source upper bound", symbolic[2])

## Execute through append

The normal append path below squares every complex input amplitude coherently, then restores the input with the exact inverse. There are zero work wires and zero measurements, so there is one coherent execution branch. Primitive CNOTs, fixed-wire binding SWAPs and declared PLU costs are printed separately.

In [ ]:
from collections import Counter
encoding = binary_field_encoding(4)
print("Mathematical square matrix", field_frobenius_matrix(encoding))
wires = (0, 1, 2, 3)
modulus = int(encoding.gf_type.irreducible_poly)
plan = gf2_frobenius(wires, modulus=modulus)
inverse = gf2_frobenius(wires, modulus=modulus, adjoint=True)
state = torch.complex(torch.arange(1, 17, dtype=torch.float64), torch.arange(16, dtype=torch.float64))
state /= state.norm()
expected = torch.empty_like(state)
for value in range(16):
    expected[int(field_frobenius_classical_value(encoding, encoding.gf_type(value)))] = state[value]
bound = bind_field_outputs(plan, (wires,))
circuit = fq.Circuit(4, inputs=state, dtype=torch.complex128).append(FieldOperation(4, bound))
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=1e-12)
restored = circuit.append(FieldOperation(4, bind_field_outputs(inverse, (wires,))))
torch.testing.assert_close(restored.state().flatten(), state, rtol=0, atol=1e-12)
assert plan.scratch == inverse.scratch == ()
print("Actual PLU primitives", dict(Counter(g.name for g in plan.instructions)))
print("Actual output binding primitives", dict(Counter(g.name for g in bound[len(plan.instructions):])))
print("Declared PLU model", concrete[2])
